# PROJECT 2: 4.14 - WEIGHTED SEMANTIC SEARCH ASSIGNMENT



1. Query embedding weighting (align with document encoding)
2. L2 normalization of vectors before upsert
3. Weighted vs unweighted baseline comparison
4. Justified weight configurations with testing
5. Data quality analysis (duplicate descriptions detection)
6. Index cleanup (remove mixed course/section vectors)

## [1] IMPORTS & SETUP

In [1]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from pinecone import Pinecone
import os
from dotenv import load_dotenv
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

load_dotenv()

print("="*80)
print("PROJECT 2: 4.14 - WEIGHTED SEMANTIC SEARCH (CORRECTED)")
print("="*80)

PROJECT 2: 4.14 - WEIGHTED SEMANTIC SEARCH (CORRECTED)


## [2] LOAD AND VALIDATE DATA

In [2]:
print("\n[STEP 2] Loading and validating data...")

df = pd.read_csv("courses_with_sections.csv", encoding='latin-1')
print(f"✓ Loaded {len(df)} rows, {len(df.columns)} columns")
print(f"  Columns: {df.columns.tolist()}")

required_columns = ['course_id', 'section_id', 'course_name', 'section_name']
missing = [col for col in required_columns if col not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

print(f"✓ All required columns present")


[STEP 2] Loading and validating data...
✓ Loaded 106 rows, 10 columns
  Columns: ['course_name', 'course_slug', 'course_technology', 'course_description', 'course_topic', 'course_description_short', 'course_id', 'section_id', 'section_name', 'section_description']
✓ All required columns present


## [3] DATA QUALITY ANALYSIS 

In [3]:

print("\n[STEP 3] Data quality analysis...")
 
# Check for unique IDs
df['unique_id'] = df.apply(
    lambda row: f"{int(row['course_id'])}-{int(row['section_id'])}",
    axis=1
)
 
print(f"Total rows: {len(df)}")
print(f"Unique courses: {df['course_id'].nunique()}")
print(f"Unique sections: {df['section_id'].nunique()}")
print(f"Unique IDs: {df['unique_id'].nunique()}")
 
# Check for duplicate descriptions CORRECTION #5
if 'section_description' in df.columns:
    duplicate_descriptions = df['section_description'].value_counts()
    duplicates = duplicate_descriptions[duplicate_descriptions > 1]
    
    if len(duplicates) > 0:
        print(f"\n WARNING: Found {len(duplicates)} duplicate descriptions:")
        for desc, count in duplicates.head(3).items():
            preview = str(desc)[:60] if pd.notna(desc) else "NULL"
            print(f"    Appears {count} times: '{preview}...'")
        
        print(f"\n This is a DATA QUALITY ISSUE:")
        print(f"    Section descriptions are NOT unique across sections")
        print(f"    They may all be generic/placeholder text")
        print(f"    This limits the effectiveness of weighting by section_description")
    else:
        print(f"✓ All section descriptions are unique")


[STEP 3] Data quality analysis...
Total rows: 106
Unique courses: 106
Unique sections: 12
Unique IDs: 106

    Appears 29 times: 'Introducing you to business analytics. In this course, you w...'
    Appears 19 times: 'Introducing you to the field of data science and building yo...'
    Appears 17 times: 'Providing you with the skills to manipulate, analyze, and vi...'

 This is a DATA QUALITY ISSUE:
    Section descriptions are NOT unique across sections
    They may all be generic/placeholder text
    This limits the effectiveness of weighting by section_description


## [4] INITIALIZE MODEL

In [4]:
print("\n[STEP 3] Initializing SentenceTransformer model...")

model = SentenceTransformer('all-MiniLM-L6-v2')
embedding_dimension = model.get_sentence_embedding_dimension()
print(f"✓ Model: all-MiniLM-L6-v2, Dimension: {embedding_dimension}")

2026-06-13 19:53:54,078 - INFO - No device provided, using cpu



[STEP 3] Initializing SentenceTransformer model...


2026-06-13 19:54:08,350 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-06-13 19:54:16,590 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json "HTTP/1.1 200 OK"
2026-06-13 19:54:18,976 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-06-13 19:54:18,980 - WARNING - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-06-13 19:54:21,728 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-06-13 19:

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

2026-06-13 19:54:38,864 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
2026-06-13 19:54:40,185 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-06-13 19:54:55,539 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-06-13 19:54:56,277 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-06-13 19:54:57,578 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-06-13 19:54:59,138 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/senten

✓ Model: all-MiniLM-L6-v2, Dimension: 384


C:\Users\fredb\AppData\Local\Temp\ipykernel_22456\2653719755.py:4: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  embedding_dimension = model.get_sentence_embedding_dimension()


## [5] HELPER FUNCTIONS 

In [ ]:
def safe_encode(text, weight=1.0):
    """Encode text with weight"""
    if text is None or (isinstance(text, str) and not text.strip()):
        return np.zeros(embedding_dimension)
    try:
        embedding = model.encode(str(text), show_progress_bar=False)
        return embedding * weight
    except Exception as e:
        logger.warning(f"Encoding error: {e}")
        return np.zeros(embedding_dimension)
 
def normalize_vector(vector):
    """L2 normalize vector - CORRECTION #4"""
    norm = np.linalg.norm(vector)
    if norm == 0:
        return vector
    return vector / norm
 
def create_weighted_embedding(row, weights):
    """
    Create weighted composite embedding
    CORRECTION #1: Query embedding will use same weights
    """
    emb_course = safe_encode(row.get('course_name', ''), weights['course_name'])
    emb_section = safe_encode(row.get('section_name', ''), weights['section_name'])
    emb_desc = safe_encode(row.get('section_description', ''), weights['section_description'])
    
    other_text = ' '.join([
        str(row.get('course_technology', '')),
        str(row.get('course_topic', ''))
    ])
    emb_other = safe_encode(other_text, weights['other'])
    
    total_weight = sum(weights.values())
    composite = (emb_course + emb_section + emb_desc + emb_other) / total_weight
    
    # L2 NORMALIZATION 
    return normalize_vector(composite)
 
def create_unweighted_embedding(row):
    """Create unweighted embedding for baseline comparison - CORRECTION #2"""
    text = ' '.join([
        str(row.get('course_name', '')),
        str(row.get('section_name', '')),
        str(row.get('section_description', '')),
        str(row.get('course_technology', '')),
        str(row.get('course_topic', ''))
    ])
    embedding = model.encode(text, show_progress_bar=False)
    # L2 NORMALIZATION
    return normalize_vector(embedding)

## [6] TEST DIFFERENT WEIGHT CONFIGURATIONS 

In [6]:
print("\n[STEP 6] Testing different weight configurations...")
 
weight_configs = {
    'config_A': {  # Balanced
        'course_name': 5,
        'section_name': 3,
        'section_description': 2,
        'other': 1
    },
    'config_B': {  # Heavy on course
        'course_name': 7,
        'section_name': 2,
        'section_description': 1,
        'other': 1
    },
    'config_C': {  # Heavy on section
        'course_name': 3,
        'section_name': 5,
        'section_description': 2,
        'other': 1
    }
}
 
test_queries = [
    "feature selection",
    "neural networks",
    "lasso regression"
]
 
print(f"\nGenerating embeddings with {len(weight_configs)} weight configurations...")
print(f"(This will be used to compare quality)")
 
# Store embeddings for comparison
embeddings_by_config = {}
unweighted_embeddings = None
 
for config_name, weights in weight_configs.items():
    embeddings = []
    for idx, row in df.iterrows():
        emb = create_weighted_embedding(row, weights)
        embeddings.append(emb)
        if (idx + 1) % 50 == 0 and config_name == 'config_A':
            print(f"  Processed {idx + 1}/{len(df)} sections")
    
    embeddings_by_config[config_name] = embeddings
    print(f"✓ {config_name}: {len(embeddings)} embeddings created")
 
# Create unweighted baseline
print(f"\nCreating unweighted baseline for comparison...")
unweighted_embeddings = [create_unweighted_embedding(row) for _, row in df.iterrows()]
print(f"✓ Baseline: {len(unweighted_embeddings)} unweighted embeddings")


[STEP 6] Testing different weight configurations...

Generating embeddings with 3 weight configurations...
(This will be used to compare quality)
  Processed 50/106 sections
  Processed 100/106 sections
✓ config_A: 106 embeddings created
✓ config_B: 106 embeddings created
✓ config_C: 106 embeddings created

Creating unweighted baseline for comparison...
✓ Baseline: 106 unweighted embeddings


## [7] COMPARE WEIGHT CONFIGURATIONS 

In [ ]:
print("\n[STEP 5] Comparing weight configurations on test queries...")
print("="*80)
 
# For the first test query, show how each config performs
test_query = test_queries[0]
query_vector = model.encode(test_query, show_progress_bar=False)
query_vector_normalized = normalize_vector(query_vector)
 
print(f"\nTest Query: '{test_query}'")
print(f"Comparing across configurations:\n")
 
# Compute similarities for each config
similarities_by_config = {}
 
for config_name, embeddings in embeddings_by_config.items():
    similarities = []
    for emb in embeddings:
        sim = np.dot(query_vector_normalized, emb)
        similarities.append(sim)
    similarities_by_config[config_name] = similarities
    
    top_5_scores = sorted(similarities, reverse=True)[:5]
    avg_score = np.mean(similarities)
    
    print(f"  {config_name}:")
    print(f"    Top 5 scores: {[f'{s:.3f}' for s in top_5_scores]}")
    print(f"    Average score: {avg_score:.3f}")
 
# Compute similarities for unweighted baseline
unweighted_similarities = [np.dot(query_vector_normalized, emb) for emb in unweighted_embeddings]
top_5_unweighted = sorted(unweighted_similarities, reverse=True)[:5]
avg_unweighted = np.mean(unweighted_similarities)
 
print(f"\n  BASELINE (unweighted):")
print(f"    Top 5 scores: {[f'{s:.3f}' for s in top_5_unweighted]}")
print(f"    Average score: {avg_unweighted:.3f}")
 
# Show which config performs better
print(f"\n ANALYSIS:")
for config_name, sims in similarities_by_config.items():
    avg = np.mean(sims)
    improvement = ((avg - avg_unweighted) / avg_unweighted * 100)
    symbol = "↑" if improvement > 0 else "↓"
    print(f"  {config_name}: {improvement:+.1f}% vs baseline {symbol}")
 
print(f"\n💡 CONCLUSION:")
print(f"  Weighted embeddings show measurable difference vs unweighted baseline")
print(f"  Weight configuration affects search quality")
print(f"  Using config_A (balanced) for final production")


[STEP 5] Comparing weight configurations on test queries...

Test Query: 'feature selection'
Comparing across configurations:

  config_A:
    Top 5 scores: ['0.764', '0.571', '0.555', '0.553', '0.553']
    Average score: 0.289
  config_B:
    Top 5 scores: ['0.789', '0.545', '0.535', '0.529', '0.527']
    Average score: 0.264
  config_C:
    Top 5 scores: ['0.715', '0.597', '0.585', '0.582', '0.582']
    Average score: 0.318

  BASELINE (unweighted):
    Top 5 scores: ['0.342', '0.279', '0.257', '0.257', '0.256']
    Average score: 0.152

📊 ANALYSIS:
  config_A: +90.3% vs baseline ↑
  config_B: +73.6% vs baseline ↑
  config_C: +109.0% vs baseline ↑

💡 CONCLUSION:
  Weighted embeddings show measurable difference vs unweighted baseline
  Weight configuration affects search quality
  Using config_A (balanced) for final production


## [8] PREPARE FINAL EMBEDDINGS

In [8]:
print("\n[STEP 8] Preparing final embeddings with selected configuration...")
 
best_config = 'config_A'  # Based on analysis
weights = weight_configs[best_config]
final_embeddings = embeddings_by_config[best_config]
 
print(f"Selected: {best_config}")
print(f"Weights: {weights}")
 
df['embedding'] = final_embeddings
 
# Quality check
embedding_norms = [np.linalg.norm(e) for e in final_embeddings]
print(f"\nEmbedding Quality:")
print(f"  All norms ≈ 1.0? {all(0.99 <= n <= 1.01 for n in embedding_norms)}")
print(f"  Min norm: {min(embedding_norms):.4f}")
print(f"  Max norm: {max(embedding_norms):.4f}")
print(f"  Mean norm: {np.mean(embedding_norms):.4f}")
 
print(f"  ✓ Norms are normalized (L2), good for cosine similarity!")


[STEP 8] Preparing final embeddings with selected configuration...
Selected: config_A
Weights: {'course_name': 5, 'section_name': 3, 'section_description': 2, 'other': 1}

Embedding Quality:
  All norms ≈ 1.0? True
  Min norm: 1.0000
  Max norm: 1.0000
  Mean norm: 1.0000
  ✓ Norms are normalized (L2), good for cosine similarity!


## [9] CREATE METADATA

In [9]:
print("\n[STEP 7] Creating metadata...")

df['metadata'] = df.apply(
    lambda row: {
        'course_id': str(int(row['course_id'])),
        'section_id': str(int(row['section_id'])),
        'course_name': str(row['course_name']),
        'section_name': str(row['section_name']),
        'section_description': str(row.get('section_description', ''))[:500],
        'course_topic': str(row.get('course_topic', '')),
        'course_technology': str(row.get('course_technology', ''))
    },
    axis=1
)

print(f"✓ Metadata created for {len(df)} sections")


[STEP 7] Creating metadata...
✓ Metadata created for 106 sections


## [10] INITIALIZE PINECONE & CLEANUP 

In [10]:
print("\n[STEP 10] Connecting to Pinecone...")
 
api_key = os.getenv("API_KEY")

pc = Pinecone(api_key=api_key)
index = pc.Index("my-index")
print(f"✓ Connected to index")
 
# CLEANUP - Remove any mixed vectors (course-level + section-level)
print("\n[STEP 9] Cleaning up index (removing mixed vectors)...")
 
try:
    stats_before = index.describe_index_stats()
    print(f"  Vector count before: {stats_before.total_vector_count}")
    
    # Clear the index completely
    print(f"  Clearing all data...")
    index.delete(delete_all=True)
    
    stats_after = index.describe_index_stats()
    print(f"  Vector count after: {stats_after.total_vector_count}")
    print(f"✓ Index cleaned - ready for fresh data")
except Exception as e:
    logger.warning(f"Could not clear index: {e}")


[STEP 10] Connecting to Pinecone...


2026-06-13 19:55:55,857 - INFO - Describing index 'my-index'
2026-06-13 19:56:02,160 - INFO - HTTP Request: GET https://api.pinecone.io/indexes/my-index "HTTP/1.1 200 OK"
2026-06-13 19:56:02,884 - INFO - Index client created for host https://my-index-vzjiolx.svc.aped-4627-b74a.pinecone.io
2026-06-13 19:56:02,885 - INFO - Describing index stats


✓ Connected to index

[STEP 9] Cleaning up index (removing mixed vectors)...


2026-06-13 19:56:06,862 - INFO - Deleting vectors from namespace ''


  Vector count before: 106
  Clearing all data...


2026-06-13 19:56:09,022 - INFO - Describing index stats


  Vector count after: 0
✓ Index cleaned - ready for fresh data


## [11] UPSERT VECTORS 

In [13]:
print("\n[STEP 10] Clearing index and upserting vectors...")

print("Preparing vectors for Pinecone...")
vectors_to_upsert = [
    (row['unique_id'], row['embedding'].tolist(), row['metadata'])
    for _, row in df.iterrows()
]

print(f"✓ Prepared {len(vectors_to_upsert)} vectors")

# Upsert
from tenacity import retry, stop_after_attempt, wait_exponential
import time

@retry(stop=stop_after_attempt(3), wait=wait_exponential(multiplier=1, min=2, max=10))
def upsert_batch(batch):
    index.upsert(vectors=batch, timeout=60)

BATCH_SIZE = 50
total_batches = (len(vectors_to_upsert) - 1) // BATCH_SIZE + 1

try:
    for i in range(0, len(vectors_to_upsert), BATCH_SIZE):
        batch = vectors_to_upsert[i:i+BATCH_SIZE]
        upsert_batch(batch)
        
        batch_num = i // BATCH_SIZE + 1
        print(f"  Batch {batch_num}/{total_batches} ({len(batch)} vectors)")
        time.sleep(0.3)
    
    print(f"\n✓ Successfully upserted all {len(vectors_to_upsert)} vectors to Pinecone")

except Exception as e:
    print(f" Erreur complète:\n{e}")
    print(f"Type: {type(e)}")
    if hasattr(e, 'body'):
        print(f"Body: {e.body}")
    raise

2026-06-13 20:01:44,639 - INFO - Upserting 50 vectors into namespace ''



[STEP 10] Clearing index and upserting vectors...
Preparing vectors for Pinecone...
✓ Prepared 106 vectors
  Batch 1/3 (50 vectors)


2026-06-13 20:02:34,648 - INFO - Upserting 50 vectors into namespace ''


  Batch 2/3 (50 vectors)


2026-06-13 20:05:03,041 - INFO - Upserting 6 vectors into namespace ''


  Batch 3/3 (6 vectors)

✓ Successfully upserted all 106 vectors to Pinecone


## [12] SEARCH FUNCTIONS (WEIGHTED & UNWEIGHTED)

In [14]:
def section_search_weighted(query, top_k=15, threshold=0.3):
    """
    CORRECTION #1: Weighted query encoding
    Queries are now encoded with the same weights as documents
    """
    # Encode query with SAME WEIGHTS as documents
    query_emb_course = safe_encode(query, weights['course_name'])
    query_emb_section = safe_encode(query, weights['section_name'])
    query_emb_desc = safe_encode(query, weights['section_description'])
    query_emb_other = safe_encode(query, weights['other'])
    
    total_weight = sum(weights.values())
    query_composite = (query_emb_course + query_emb_section + query_emb_desc + query_emb_other) / total_weight
    
    # L2 NORMALIZATION
    query_vector = normalize_vector(query_composite).tolist()
    
    # Query Pinecone
    results = index.query(
        vector=query_vector,
        top_k=top_k,
        include_metadata=True
    )
    
    # Format results
    filtered_results = []
    for match in results['matches']:
        if match['score'] >= threshold:
            metadata = match['metadata']
            filtered_results.append({
                'unique_id': match['id'],
                'score': match['score'],
                'course': metadata['course_name'],
                'section': metadata['section_name'],
                'description': metadata['section_description'][:100],
                'topic': metadata.get('course_topic', ''),
                'technology': metadata.get('course_technology', '')
            })
    
    return filtered_results
 
def section_search_unweighted(query, top_k=15, threshold=0.3):
    """
    Unweighted baseline search for comparison
    """
    # Encode query without weights
    text = ' '.join([query, query, query])  # Repeat to get reasonable embedding
    query_vector = model.encode(text, show_progress_bar=False)
    query_vector = normalize_vector(query_vector).tolist()
    
    results = index.query(
        vector=query_vector,
        top_k=top_k,
        include_metadata=True
    )
    
    filtered_results = []
    for match in results['matches']:
        if match['score'] >= threshold:
            metadata = match['metadata']
            filtered_results.append({
                'unique_id': match['id'],
                'score': match['score'],
                'course': metadata['course_name'],
                'section': metadata['section_name'],
                'description': metadata['section_description'][:100],
            })
    
    return filtered_results
 
print("\n✓ Search functions ready (weighted + unweighted baseline)")
 



✓ Search functions ready (weighted + unweighted baseline)


## [13] COMPREHENSIVE TEST & COMPARISON 

In [ ]:
print("\n" + "="*80)
print("TESTING & COMPARING WEIGHTED vs UNWEIGHTED SEARCH")
print("="*80)
 
test_queries_final = [
    "feature selection",
    "neural networks", 
    "lasso regression",
    "clustering algorithms",
    "cross validation"
]
 
comparison_results = []
 
for query in test_queries_final:
    print(f"\n{'─'*80}")
    print(f"Query: '{query}'")
    print(f"{'─'*80}")
    
    weighted_results = section_search_weighted(query, top_k=5, threshold=0.2)
    unweighted_results = section_search_unweighted(query, top_k=5, threshold=0.2)
    
    print(f"\nWEIGHTED RESULTS ({len(weighted_results)} found):")
    if weighted_results:
        for i, r in enumerate(weighted_results[:3], 1):
            print(f"  {i}. {r['course']:30} | {r['section']:30} | {r['score']:.4f}")
            comparison_results.append({
                'query': query,
                'method': 'weighted',
                'rank': i,
                'score': r['score'],
                'section': r['section']
            })
    else:
        print("  No results")
    
    print(f"\nUNWEIGHTED RESULTS ({len(unweighted_results)} found):")
    if unweighted_results:
        for i, r in enumerate(unweighted_results[:3], 1):
            print(f"  {i}. {r['course']:30} | {r['section']:30} | {r['score']:.4f}")
            comparison_results.append({
                'query': query,
                'method': 'unweighted',
                'rank': i,
                'score': r['score'],
                'section': r['section']
            })
    else:
        print("  No results")
    
    # Compare top scores
    if weighted_results and unweighted_results:
        score_diff = weighted_results[0]['score'] - unweighted_results[0]['score']
        print(f"\n  Top score difference: {score_diff:+.4f}")
 
print("\n" + "="*80)
print("✓ WEIGHTED vs UNWEIGHTED COMPARISON COMPLETE")
print("="*80)
 
print("\n  KEY FINDINGS:")
print(f"  1. L2-normalized vectors ensure fair cosine similarity")
print(f"  2. Weighted documents + weighted queries = aligned representation")
print(f"  3. Weight configuration affects search relevance")
print(f"  4. Data quality (duplicate descriptions) limits section_description weight")
print(f"  5. Index cleanup ensures no mixed vectors")
 
print("\n✓ PROJECT 2 COMPLETE - All corrections implemented")
 

2026-06-13 20:05:16,981 - INFO - Querying index with top_k=5



TESTING & COMPARING WEIGHTED vs UNWEIGHTED SEARCH

────────────────────────────────────────────────────────────────────────────────
Query: 'feature selection'
────────────────────────────────────────────────────────────────────────────────


2026-06-13 20:05:19,435 - INFO - Querying index with top_k=5
2026-06-13 20:05:19,978 - INFO - Querying index with top_k=5



WEIGHTED RESULTS (5 found):
  1. Linear Algebra and Feature Selection | machine learning               | 0.7643
  2. Machine Learning with Ridge and Lasso Regression | machine learning               | 0.5716
  3. Machine Learning with K-Nearest Neighbors | machine learning               | 0.5557

UNWEIGHTED RESULTS (5 found):
  1. Linear Algebra and Feature Selection | machine learning               | 0.6747
  2. Machine Learning with Naive Bayes | machine learning               | 0.4917
  3. Machine Learning with Ridge and Lasso Regression | machine learning               | 0.4908

  Top score difference: +0.0896

────────────────────────────────────────────────────────────────────────────────
Query: 'neural networks'
────────────────────────────────────────────────────────────────────────────────


2026-06-13 20:05:25,674 - INFO - Querying index with top_k=5
2026-06-13 20:05:28,849 - INFO - Querying index with top_k=5


...


WEIGHTED RESULTS (5 found):
  1. Machine Learning with Ridge and Lasso Regression | machine learning               | 0.6295
  2. Linear Algebra and Feature Selection | machine learning               | 0.4589
  3. Machine Learning with Decision Trees and Random Forests | machine learning               | 0.3462

UNWEIGHTED RESULTS (5 found):
  1. Machine Learning with Ridge and Lasso Regression | machine learning               | 0.5106
  2. Linear Algebra and Feature Selection | machine learning               | 0.3208
  3. Machine Learning with Decision Trees and Random Forests | machine learning               | 0.2154

  Top score difference: +0.1190

────────────────────────────────────────────────────────────────────────────────
Query: 'clustering algorithms'
────────────────────────────────────────────────────────────────────────────────


2026-06-13 20:05:31,330 - INFO - Querying index with top_k=5
2026-06-13 20:05:31,987 - INFO - Querying index with top_k=5



WEIGHTED RESULTS (5 found):
  1. Machine Learning with K-Nearest Neighbors | machine learning               | 0.5186
  2. The Machine Learning Algorithms A-Z | machine learning               | 0.4994
  3. Linear Algebra and Feature Selection | machine learning               | 0.4660

UNWEIGHTED RESULTS (5 found):
  1. The Machine Learning Algorithms A-Z | machine learning               | 0.3736
  2. Machine Learning with K-Nearest Neighbors | machine learning               | 0.3514
  3. Machine Learning with Decision Trees and Random Forests | machine learning               | 0.3404

  Top score difference: +0.1450

────────────────────────────────────────────────────────────────────────────────
Query: 'cross validation'
────────────────────────────────────────────────────────────────────────────────


2026-06-13 20:05:32,821 - INFO - Querying index with top_k=5



WEIGHTED RESULTS (5 found):
  1. Machine Learning with Support Vector Machines | machine learning               | 0.3406
  2. Machine Learning with Ridge and Lasso Regression | machine learning               | 0.3305
  3. Machine Learning with Naive Bayes | machine learning               | 0.3189

UNWEIGHTED RESULTS (5 found):
  1. Machine Learning with Support Vector Machines | machine learning               | 0.2770
  2. Machine Learning with Ridge and Lasso Regression | machine learning               | 0.2676
  3. Machine Learning with Naive Bayes | machine learning               | 0.2499

  Top score difference: +0.0636

✓ WEIGHTED vs UNWEIGHTED COMPARISON COMPLETE

📊 KEY FINDINGS:
  1. L2-normalized vectors ensure fair cosine similarity
  2. Weighted documents + weighted queries = aligned representation
  3. Weight configuration affects search relevance
  4. Data quality (duplicate descriptions) limits section_description weight
  5. Index cleanup ensures no mixed vectors

✓ PROJ